In [ ]:
# -*- coding: utf-8 -*-
"""
Este notebook Jupyter demonstra como construir um sistema de recomendação de relógios com base em uma imagem de entrada.

O sistema irá:
1.  Simular um conjunto de dados de 4 classes de relógios.
2.  Usar um modelo de Transfer Learning (MobileNetV2) para extrair embeddings.
3.  Treinar um modelo simples para classificar a imagem de entrada em uma das 4 classes.
4.  Construir um índice de busca eficiente (usando Annoy) para cada classe.
5.  Dada uma imagem de entrada, o sistema irá:
    - Classificá-la para determinar sua classe.
    - Buscar no índice daquela classe por relógios semelhantes.
    - Retornar os 5 relógios mais parecidos, com suas similaridades.
"""

# =================================================================================================
# 1. Configuração e Importação de Bibliotecas
# =================================================================================================

# Instalação de bibliotecas necessárias (pode ser executado diretamente no Google Colab)
!pip install tensorflow scikit-learn annoy matplotlib Pillow

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from annoy import AnnoyIndex
import os
import glob
from tqdm import tqdm
import shutil
from IPython.display import Image as DisplayImage, display
from google.colab import drive
import zipfile
import datetime

# =================================================================================================
# 2. Simulação e Pré-processamento do Dataset
# =================================================================================================

# --- Carregar Dataset do Arquivo ZIP ---
ZIP_PATH = '/content/watch_dataset.zip'
BASE_DIR = '/content/watch_dataset'

# Descompactar o arquivo ZIP para o BASE_DIR
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)
os.makedirs(BASE_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(BASE_DIR)

# Criar um DataFrame de metadados a partir da estrutura de diretórios
def load_real_metadata(base_dir):
    data = []

    # Lógica para lidar com uma possível pasta extra no diretório raiz do ZIP
    dir_content = os.listdir(base_dir)
    if len(dir_content) == 1 and os.path.isdir(os.path.join(base_dir, dir_content[0])):
        base_dir = os.path.join(base_dir, dir_content[0])
        print(f"Detectada uma pasta extra no nível superior. Novo diretório base: {base_dir}")

    classes = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    # As classes que você forneceu: pulso, bolso, mesa, parede
    CLASSES_FILTER = ['pulso', 'bolso', 'mesa', 'parede']
    classes = [c for c in classes if c in CLASSES_FILTER]

    print(f"Subpastas filtradas encontradas em {base_dir}: {classes}")

    for class_name in classes:
        class_dir = os.path.join(base_dir, class_name)
        # Procura por imagens com extensões comuns, ignorando maiúsculas e minúsculas
        image_paths = glob.glob(os.path.join(class_dir, '*.jpg')) + \
                      glob.glob(os.path.join(class_dir, '*.jpeg')) + \
                      glob.glob(os.path.join(class_dir, '*.JPG')) + \
                      glob.glob(os.path.join(class_dir, '*.PNG')) + \
                      glob.glob(os.path.join(class_dir, '*.png'))
        print(f"Imagens encontradas na pasta {class_name}: {len(image_paths)}")
        for img_path in image_paths:
            data.append({
                'image_path': img_path,
                'class': class_name,
                'id': os.path.basename(img_path).split('.')[0]
            })
    return pd.DataFrame(data), classes

# Gerar o dataset a partir do diretório
dataset, CLASSES = load_real_metadata(BASE_DIR)

print(f"Dataset carregado com {len(dataset)} imagens.")
print(f"Classes encontradas: {CLASSES}")
print("\nPrimeiras 5 linhas do dataset:")
display(dataset.head())

# --- Pré-processamento de Imagem para o modelo ---
def preprocess_image(image_path):
    """Carrega, redimensiona e pré-processa uma imagem para o modelo."""
    img = image.load_img(image_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    # MobileNetV2 espera que os valores de pixel estejam entre -1 e 1
    return tf.keras.applications.mobilenet_v2.preprocess_input(img_array)

# =================================================================================================
# 3. Modelo de Classificação e Embeddings
# =================================================================================================

# Usamos a técnica de Transfer Learning para reaproveitar uma rede neural já treinada.
# O MobileNetV2 é uma boa escolha por ser leve e eficiente.

# Parte 1: Modelo para extrair os vetores de características (embeddings)
# Carrega o MobileNetV2 pré-treinado no ImageNet, mas sem a camada de classificação final.
embedding_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
# Adicionar uma camada de pooling para achatar a saída do MobileNetV2 para um vetor 1D
embedding_model = Model(
    inputs=embedding_model.input,
    outputs=GlobalAveragePooling2D()(embedding_model.output)
)
print("Modelo de Embeddings (MobileNetV2) carregado e pronto.")

# Parte 2: Modelo para Classificação
# Adicionamos uma "cabeça" de classificação no topo do modelo de embeddings.
num_classes = len(CLASSES)
x = embedding_model.output
x = Dropout(0.2)(x)
x = Dense(num_classes, activation='softmax')(x)
classification_model = Model(inputs=embedding_model.input, outputs=x)

classification_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Modelo de Classificação criado.")
print("\nResumo do modelo de classificação:")
classification_model.summary()

# Nota: Para um sistema real, este modelo de classificação precisaria ser
# treinado com os seus dados rotulados. Neste notebook, vamos simular a
# classificação para manter a demonstração simples.

# =================================================================================================
# 4. Construção do Índice de Similaridade (Annoy)
# =================================================================================================

# Criamos um índice de busca eficiente (Annoy) para cada classe.
# Isso torna a busca muito mais rápida do que comparar a imagem de entrada com todas as imagens do dataset.

class_indexes = {}
class_data = {}
embedding_dim = embedding_model.output_shape[1] # Dimensão do vetor de saída do MobileNetV2

# Verifica se o dataset não está vazio antes de construir os índices
if not dataset.empty:
    for class_name in tqdm(CLASSES, desc="Construindo índices Annoy"):
        # Filtrar o dataset por classe
        class_df = dataset[dataset['class'] == class_name].reset_index(drop=True)

        # Gerar embeddings para as imagens desta classe
        class_embeddings = np.array([
            embedding_model.predict(preprocess_image(path)) for path in class_df['image_path']
        ]).squeeze()

        # Criar um novo índice Annoy para a classe
        t = AnnoyIndex(embedding_dim, 'angular')
        for i, embedding in enumerate(class_embeddings):
            t.add_item(i, embedding)

        t.build(50)  # 50 árvores para um bom equilíbrio entre velocidade e precisão
        class_indexes[class_name] = t
        class_data[class_name] = class_df

    print("\nÍndices de similaridade Annoy criados para todas as classes.")
else:
    print("\nO dataset está vazio. Não foi possível construir os índices Annoy.")

# =================================================================================================
# 5. Função Principal: Classificar e Buscar Relógios Similares
# =================================================================================================

def find_similar_clocks(input_image_path, top_k=5):
    """
    Classifica a imagem de entrada, busca relógios similares na classe predita e os retorna.
    """
    print("Iniciando a busca por relógios similares...")

    # 1. Pré-processar e obter o embedding da imagem de entrada
    input_preprocessed = preprocess_image(input_image_path)
    input_embedding = embedding_model.predict(input_preprocessed)

    # 2. Classificar a imagem de entrada (agora de forma consistente)
    # Em um sistema real, este passo usaria o modelo treinado.
    # Para a demonstração, extraímos a classe real do caminho do arquivo.
    actual_class = dataset[dataset['image_path'] == input_image_path]['class'].iloc[0]
    predicted_class = actual_class

    print(f"\nA imagem de entrada foi classificada como: '{predicted_class}'")

    # 3. Usar o índice da classe predita para encontrar vizinhos
    class_index = class_indexes[predicted_class]
    nns_indices = class_index.get_nns_by_vector(input_embedding[0], n=top_k, include_distances=True)

    similar_items_df = class_data[predicted_class].iloc[nns_indices[0]].copy()

    # 4. Adicionar a similaridade de cosseno aos resultados
    input_embedding_flat = input_embedding.reshape(1, -1)
    similar_embeddings = np.array([
        embedding_model.predict(preprocess_image(path)) for path in similar_items_df['image_path']
    ]).squeeze()

    similar_items_df['cosine_similarity'] = cosine_similarity(input_embedding_flat, similar_embeddings).flatten()

    # 5. Visualizar a imagem de entrada e os resultados
    print(f"\n{top_k} relógios mais parecidos da classe '{predicted_class}':")

    fig, axes = plt.subplots(1, top_k + 1, figsize=(20, 4))

    # Imagem de entrada
    axes[0].imshow(Image.open(input_image_path))
    axes[0].set_title("Imagem de Entrada", fontsize=10)
    axes[0].axis('off')

    # Relógios similares encontrados
    for i, (index, row) in enumerate(similar_items_df.iterrows()):
        img_path = row['image_path']
        similarity = row['cosine_similarity']

        axes[i + 1].imshow(Image.open(img_path))
        axes[i + 1].set_title(f"Sim: {similarity:.2f}", fontsize=10)
        axes[i + 1].axis('off')

    plt.tight_layout()

    # Salvar e exibir o gráfico no Google Drive com um nome de arquivo único
    output_dir = '/content/drive/MyDrive/relogios_similares_saida/'
    os.makedirs(output_dir, exist_ok=True)

    # Cria um nome de arquivo único usando a data e hora atuais
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f'similar_clocks_{os.path.basename(input_image_path).split(".")[0]}_{timestamp}.png'
    output_file = os.path.join(output_dir, output_filename)

    try:
        # Salva o gráfico em um arquivo PNG
        plt.savefig(output_file)
        plt.close(fig) # Fecha o gráfico para evitar duplicatas no output

        # Exibe a imagem salva no Canvas a partir do Google Drive
        print(f"\nGráfico salvo e exibido de: {output_file}")
        display(DisplayImage(output_file))
    except Exception as e:
        print(f"Erro ao salvar ou exibir o gráfico: {e}")
        # Se houver um erro, ainda tentamos exibir a figura diretamente, caso o ambiente suporte
        display(fig)

    # Imprimir os relógios classificados como semelhantes
    print("\nDetalhes dos relógios similares:")
    display(similar_items_df[['id', 'class', 'cosine_similarity']])

    return similar_items_df

# =================================================================================================
# 6. Demonstração
# =================================================================================================

# Montar o Google Drive para salvar os resultados
drive.mount('/content/drive')

# A lista de classes agora é carregada dinamicamente
# Escolha uma imagem de entrada aleatória de uma classe aleatória
chosen_class = np.random.choice(CLASSES)
print(f"Escolhendo uma imagem de exemplo da classe: {chosen_class}")

# Se não houver imagens para a classe escolhida, escolha outra classe
if dataset[dataset['class'] == chosen_class].empty:
  print(f"A classe '{chosen_class}' não possui imagens. Escolhendo outra classe.")
  CLASSES.remove(chosen_class)
  if not CLASSES:
    print("Não há classes com imagens no dataset. Por favor, verifique o diretório.")
  else:
    chosen_class = np.random.choice(CLASSES)
    print(f"Nova classe escolhida: {chosen_class}")

# Garante que o dataset não esteja vazio antes de tentar selecionar uma imagem
if not dataset.empty and chosen_class in dataset['class'].values:
  # Seleciona uma imagem aleatória da classe escolhida
  sample_input_path = dataset[dataset['class'] == chosen_class]['image_path'].sample(n=1).iloc[0]
  print(f"Imagem de entrada escolhida: {sample_input_path}")

  # Chame a função principal para encontrar relógios similares
  similar_clocks = find_similar_clocks(sample_input_path, top_k=5)
else:
  print("Não foi possível encontrar uma imagem de exemplo para demonstração. Verifique o caminho do dataset.")

